<a href="https://colab.research.google.com/github/Tejaswimadastu/Generative_AI/blob/main/FAISS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FAISS

### Load Data

In [1]:
chunks = [

    "Employees receive 12 casual leaves annually.",

    "Employees receive 15 sick leaves annually.",

    "Employees may work from home twice per week.",

    "Travel expenses are reimbursed within 30 days.",

    "All employees are covered under company medical insurance."

]

print("Total Chunks:", len(chunks))

Total Chunks: 5


### Load Embedding Model

In [2]:
from sentence_transformers import SentenceTransformer
model=SentenceTransformer('all-MiniLM-L6-v2')
print("Embedding model loaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded


### Generate Embeddings

In [4]:
embeddings=model.encode(chunks)
print("Embeddings generated")
print(embeddings.shape)

Embeddings generated
(5, 384)


### Create FAISS Index

In [5]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 32.0 MB/s eta 0:00:00


In [6]:
import faiss
dimension=embeddings.shape[1]
index=faiss.IndexFlatL2(dimension)
print("FAISS index created")

FAISS index created


### Add Embeddings to FAISS

In [8]:
import numpy as np

from numpy import dtype
index.add(
    np.array(embeddings,dtype=np.float32)
)
print("Vectors Stored:",index.ntotal)

Vectors Stored: 10


### User Query

In [10]:
query="How many casual leaves are allowed?"
print(query)

How many casual leaves are allowed?


### Convert query into Embeddings

In [20]:
query_embedding=model.encode([query])


### Search Faiss

In [21]:
k=3
distances,indices=index.search(
    np.array(
        query_embedding,
        dtype=np.float32
    ),
    k
    )


### Display Results

In [23]:
print("Machting Chunks:")
for rank,idx in enumerate(indices[0]):
  print(f"Rank {rank+1}:")
  print(chunks[idx % len(chunks)])
  print(f"Distance: {distances[0][rank]}")
  print("-"*50)

Machting Chunks:
Rank 1:
Employees receive 12 casual leaves annually.
Distance: 0.6896418333053589
--------------------------------------------------
Rank 2:
Employees receive 12 casual leaves annually.
Distance: 0.6896418333053589
--------------------------------------------------
Rank 3:
Employees receive 15 sick leaves annually.
Distance: 1.1779993772506714
--------------------------------------------------


### Intelligent Employee Policy Assistant using RAG
### Problem Statement
A multinational company maintains thousands of employee-related documents, including HR policies, leave regulations, travel reimbursement guidelines, medical insurance policies, and work-from-home procedures.
Employees frequently ask questions about company policies. Searching through multiple PDF documents manually is time-consuming and inefficient.
Your task is to build a Retrieval-Augmented Generation (RAG) system that can answer employee questions accurately by retrieving relevant information from company policy documents and generating human-readable responses using a Large Language Model.
Dataset
Create and use the following PDFs:
Employee Handbook.pdf
Leave Policy.pdf
Travel Policy.pdf
Work From Home Policy.pdf
Medical Insurance Policy.pdf
You may create sample content or use real HR policy documents.
Task 1: Document Loading
Objective
Load all company policy PDFs.
Requirements
Read multiple PDF files.
Extract text content.
Display document statistics.  
Deliverables
Display:
File Name
Number of Pages
Total Characters
Total Words
Task 2: Document Chunking
Objective
Split large policy documents into smaller chunks.
Requirements
Implement:
Fixed Size Chunking
Recursive Chunking
Use:
Chunk Size = 500
Chunk Overlap = 100
Deliverables
Display:
Chunk Number
Chunk Content
Chunk Length
Analysis
Explain:
Why chunking is necessary in RAG.


In [24]:
!pip install pymupdf
!pip install langchain-text-splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 34.2 MB/s eta 0:00:00


In [25]:
import fitz
import pandas as pd
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [26]:
pdf_files = [
    "Employee_Handbook.pdf",
    "Leave_Policy.pdf",
    "Travel_Policy.pdf",
    "Work_From_Home_Policy.pdf",
    "Medical_Insurance_Policy.pdf"
]

In [28]:
documents = []
stats = []

# Create dummy PDF files if they don't exist
for file_name in pdf_files:
    try:
        with open(file_name, 'r') as f:
            # File exists, do nothing
            pass
    except FileNotFoundError:
        print(f"Creating dummy file: {file_name}")
        # Create a simple dummy PDF using fitz
        doc = fitz.open()
        page = doc.new_page()
        page.insert_text(fitz.Point(50, 50), f"This is a dummy content for {file_name}")
        doc.save(file_name)
        doc.close()

for file in pdf_files:

    pdf = fitz.open(file)

    text = ""

    for page in pdf:
        text += page.get_text()

    documents.append({
        "file_name": file,
        "content": text
    })

    stats.append([
        file,
        len(pdf),
        len(text),
        len(text.split())
    ])

Creating dummy file: Employee_Handbook.pdf
Creating dummy file: Leave_Policy.pdf
Creating dummy file: Travel_Policy.pdf
Creating dummy file: Work_From_Home_Policy.pdf
Creating dummy file: Medical_Insurance_Policy.pdf


In [29]:
df = pd.DataFrame(
    stats,
    columns=[
        "File Name",
        "Number of Pages",
        "Total Characters",
        "Total Words"
    ]
)

print(df)

                      File Name  Number of Pages  Total Characters  \
0         Employee_Handbook.pdf                1                50   
1              Leave_Policy.pdf                1                45   
2             Travel_Policy.pdf                1                46   
3     Work_From_Home_Policy.pdf                1                54   
4  Medical_Insurance_Policy.pdf                1                57   

   Total Words  
0            7  
1            7  
2            7  
3            7  
4            7  


In [30]:
def fixed_size_chunking(text, chunk_size=500):

    chunks = []

    for i in range(0, len(text), chunk_size):
        chunks.append(text[i:i+chunk_size])

    return chunks

In [31]:
all_fixed_chunks = []

for doc in documents:

    chunks = fixed_size_chunking(
        doc["content"],
        chunk_size=500
    )

    all_fixed_chunks.extend(chunks)

In [32]:
print("="*60)
print("FIXED SIZE CHUNKING")
print("="*60)

for i, chunk in enumerate(all_fixed_chunks, start=1):

    print("\n")
    print("Chunk Number :", i)

    print(chunk)

    print("\nChunk Length :", len(chunk))

    print("-"*60)

FIXED SIZE CHUNKING


Chunk Number : 1
This is a dummy content for Employee_Handbook.pdf


Chunk Length : 50
------------------------------------------------------------


Chunk Number : 2
This is a dummy content for Leave_Policy.pdf


Chunk Length : 45
------------------------------------------------------------


Chunk Number : 3
This is a dummy content for Travel_Policy.pdf


Chunk Length : 46
------------------------------------------------------------


Chunk Number : 4
This is a dummy content for Work_From_Home_Policy.pdf


Chunk Length : 54
------------------------------------------------------------


Chunk Number : 5
This is a dummy content for Medical_Insurance_Policy.pdf


Chunk Length : 57
------------------------------------------------------------


In [33]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

In [34]:
all_recursive_chunks = []

for doc in documents:

    chunks = splitter.split_text(
        doc["content"]
    )

    all_recursive_chunks.extend(chunks)

In [35]:
print("="*60)
print("RECURSIVE CHUNKING")
print("="*60)

for i, chunk in enumerate(all_recursive_chunks, start=1):

    print("\n")
    print("Chunk Number :", i)

    print(chunk)

    print("\nChunk Length :", len(chunk))

    print("-"*60)

RECURSIVE CHUNKING


Chunk Number : 1
This is a dummy content for Employee_Handbook.pdf

Chunk Length : 49
------------------------------------------------------------


Chunk Number : 2
This is a dummy content for Leave_Policy.pdf

Chunk Length : 44
------------------------------------------------------------


Chunk Number : 3
This is a dummy content for Travel_Policy.pdf

Chunk Length : 45
------------------------------------------------------------


Chunk Number : 4
This is a dummy content for Work_From_Home_Policy.pdf

Chunk Length : 53
------------------------------------------------------------


Chunk Number : 5
This is a dummy content for Medical_Insurance_Policy.pdf

Chunk Length : 56
------------------------------------------------------------


#Task 4: Build FAISS Vector Database
Objective
Store embeddings efficiently.
Requirements
Create FAISS index.
Insert chunk embeddings.
Deliverables
Display:
Number of Chunks
Number of Stored Vectors
Embedding Dimension


In [37]:
!pip install faiss-cpu

In [38]:
import faiss
import numpy as np

In [39]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_embeddings = model.encode(all_recursive_chunks)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [40]:
chunk_embeddings = np.array(
    chunk_embeddings,
    dtype="float32"
)

In [41]:
embedding_dimension = chunk_embeddings.shape[1]

index = faiss.IndexFlatL2(
    embedding_dimension
)

index.add(chunk_embeddings)

In [42]:
print("=" * 50)
print("FAISS VECTOR DATABASE")
print("=" * 50)

print("Number of Chunks :",
      len(all_recursive_chunks))

print("Number of Stored Vectors :",
      index.ntotal)

print("Embedding Dimension :",
      embedding_dimension)

FAISS VECTOR DATABASE
Number of Chunks : 5
Number of Stored Vectors : 5
Embedding Dimension : 384


In [43]:
print("FAISS Index Created Successfully")
print("Total Vectors Stored :", index.ntotal)

FAISS Index Created Successfully
Total Vectors Stored : 5


### Task 5: Semantic Retrieval Objective
#### Retrieve relevant chunks for employee questions.
Example Queries
How many casual leaves are available?
Can employees work from home?
How does travel reimbursement work?
Who is covered under medical insurance?
Deliverables
Display:
Query
Top 3 Retrieved Chunks
Similarity Scores
Analysis
Explain:
Why those chunks were retrieved.


In [44]:
queries = [
    "How many casual leaves are available?",
    "Can employees work from home?",
    "How does travel reimbursement work?",
    "Who is covered under medical insurance?"
]

In [45]:
query_embeddings = model.encode(queries)

query_embeddings = np.array(
    query_embeddings,
    dtype="float32"
)

In [46]:
top_k = 3

for i, query in enumerate(queries):

    print("\n")
    print("=" * 70)

    print("Query:")
    print(query)

    print("=" * 70)

    distances, indices = index.search(
        query_embeddings[i].reshape(1, -1),
        top_k
    )

    for rank in range(top_k):

        chunk_index = indices[0][rank]

        similarity_score = 1 / (
            1 + distances[0][rank]
        )

        print(f"\nRank {rank+1}")

        print("Similarity Score:",
              round(similarity_score, 4))

        print("\nRetrieved Chunk:")

        print(all_recursive_chunks[chunk_index])

        print("-" * 60)



Query:
How many casual leaves are available?

Rank 1
Similarity Score: 0.3559

Retrieved Chunk:
This is a dummy content for Employee_Handbook.pdf
------------------------------------------------------------

Rank 2
Similarity Score: 0.3533

Retrieved Chunk:
This is a dummy content for Leave_Policy.pdf
------------------------------------------------------------

Rank 3
Similarity Score: 0.3459

Retrieved Chunk:
This is a dummy content for Travel_Policy.pdf
------------------------------------------------------------


Query:
Can employees work from home?

Rank 1
Similarity Score: 0.4202

Retrieved Chunk:
This is a dummy content for Work_From_Home_Policy.pdf
------------------------------------------------------------

Rank 2
Similarity Score: 0.3943

Retrieved Chunk:
This is a dummy content for Employee_Handbook.pdf
------------------------------------------------------------

Rank 3
Similarity Score: 0.3489

Retrieved Chunk:
This is a dummy content for Leave_Policy.pdf
-------------

In [47]:
from sklearn.metrics.pairwise import cosine_similarity
for query in queries:

    print("\n")
    print("=" * 70)

    print("Query:")
    print(query)

    print("=" * 70)

    query_embedding = model.encode([query])

    similarities = cosine_similarity(
        query_embedding,
        chunk_embeddings
    )[0]

    top_indices = similarities.argsort()[-3:][::-1]

    for rank, idx in enumerate(top_indices):

        print(f"\nRank {rank+1}")

        print(
            "Similarity Score:",
            round(similarities[idx], 4)
        )

        print("\nRetrieved Chunk:")

        print(all_recursive_chunks[idx])

        print("-" * 60)



Query:
How many casual leaves are available?

Rank 1
Similarity Score: 0.0951

Retrieved Chunk:
This is a dummy content for Employee_Handbook.pdf
------------------------------------------------------------

Rank 2
Similarity Score: 0.0848

Retrieved Chunk:
This is a dummy content for Leave_Policy.pdf
------------------------------------------------------------

Rank 3
Similarity Score: 0.0544

Retrieved Chunk:
This is a dummy content for Travel_Policy.pdf
------------------------------------------------------------


Query:
Can employees work from home?

Rank 1
Similarity Score: 0.3101

Retrieved Chunk:
This is a dummy content for Work_From_Home_Policy.pdf
------------------------------------------------------------

Rank 2
Similarity Score: 0.2318

Retrieved Chunk:
This is a dummy content for Employee_Handbook.pdf
------------------------------------------------------------

Rank 3
Similarity Score: 0.067

Retrieved Chunk:
This is a dummy content for Leave_Policy.pdf
--------------

## Task 6: Integrate Large Language Model.

In [48]:
!pip install -q transformers accelerate torch

In [49]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    max_new_tokens=150
)

print("LLM Loaded Successfully")

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


LLM Loaded Successfully


In [50]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def retrieve_context(query, top_k=3):

    query_embedding = model.encode([query])

    similarities = cosine_similarity(
        query_embedding,
        chunk_embeddings
    )[0]

    top_indices = similarities.argsort()[-top_k:][::-1]

    retrieved_chunks = []

    for idx in top_indices:
        retrieved_chunks.append(
            all_recursive_chunks[idx]
        )

    return "\n".join(retrieved_chunks)

In [51]:
def build_prompt(context, question):

    prompt = f"""
Context:
{context}

Question:
{question}

Answer:
"""

    return prompt

In [52]:
def ask_policy_assistant(question):

    context = retrieve_context(question)

    prompt = build_prompt(
        context,
        question
    )

    response = generator(
        prompt,
        do_sample=False
    )

    answer = response[0]["generated_text"]

    return context, answer

In [53]:
question = "How many casual leaves are available?"

context, answer = ask_policy_assistant(
    question
)

print("="*60)
print("QUESTION")
print("="*60)
print(question)

print("\n")
print("="*60)
print("RETRIEVED CONTEXT")
print("="*60)
print(context)

print("\n")
print("="*60)
print("GENERATED ANSWER")
print("="*60)
print(answer)

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


QUESTION
How many casual leaves are available?


RETRIEVED CONTEXT
This is a dummy content for Employee_Handbook.pdf
This is a dummy content for Leave_Policy.pdf
This is a dummy content for Travel_Policy.pdf


GENERATED ANSWER

Context:
This is a dummy content for Employee_Handbook.pdf
This is a dummy content for Leave_Policy.pdf
This is a dummy content for Travel_Policy.pdf

Question:
How many casual leaves are available?

Answer:
There are 3 casual leaves available.


In [54]:
queries = [
    "How many casual leaves are available?",
    "Can employees work from home?",
    "How does travel reimbursement work?",
    "Who is covered under medical insurance?"
]

for query in queries:

    context, answer = ask_policy_assistant(
        query
    )

    print("\n")
    print("="*80)

    print("Query:")
    print(query)

    print("\nRetrieved Context:")
    print(context)

    print("\nGenerated Answer:")
    print(answer)

    print("="*80)

[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)




Query:
How many casual leaves are available?

Retrieved Context:
This is a dummy content for Employee_Handbook.pdf
This is a dummy content for Leave_Policy.pdf
This is a dummy content for Travel_Policy.pdf

Generated Answer:

Context:
This is a dummy content for Employee_Handbook.pdf
This is a dummy content for Leave_Policy.pdf
This is a dummy content for Travel_Policy.pdf

Question:
How many casual leaves are available?

Answer:
There are 3 casual leaves available.


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)




Query:
Can employees work from home?

Retrieved Context:
This is a dummy content for Work_From_Home_Policy.pdf
This is a dummy content for Employee_Handbook.pdf
This is a dummy content for Leave_Policy.pdf

Generated Answer:

Context:
This is a dummy content for Work_From_Home_Policy.pdf
This is a dummy content for Employee_Handbook.pdf
This is a dummy content for Leave_Policy.pdf

Question:
Can employees work from home?

Answer:
Yes, employees can work from home.


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)




Query:
How does travel reimbursement work?

Retrieved Context:
This is a dummy content for Travel_Policy.pdf
This is a dummy content for Medical_Insurance_Policy.pdf
This is a dummy content for Work_From_Home_Policy.pdf

Generated Answer:

Context:
This is a dummy content for Travel_Policy.pdf
This is a dummy content for Medical_Insurance_Policy.pdf
This is a dummy content for Work_From_Home_Policy.pdf

Question:
How does travel reimbursement work?

Answer:
Travel reimbursement is a policy that allows employees to reimburse themselves for travel expenses incurred for business purposes. This policy is designed to reduce the financial burden of travel expenses for employees and promote efficient use of company resources.

To reimburse travel expenses, employees must submit a travel expense report to their manager or HR representative. The report should include details such as the date of the trip, the purpose of the trip, the amount of the trip, and any necessary expenses such as airfa

## Task 7: Complete RAG Pipeline
Objective

Build a reusable Employee Policy Assistant.

Input
Employee Question

Internal Flow

Question

↓

Embedding

↓

FAISS Search

↓

Top-K Chunks

↓

LLM

↓

Answer

Output

Final Human-Readable Answer

In [55]:
def employee_policy_assistant(question):

    # Step 1: Convert Question to Embedding
    query_embedding = model.encode([question])

    # Step 2: Search FAISS Index
    distances, indices = index.search(
        np.array(query_embedding, dtype="float32"),
        k=3
    )

    # Step 3: Retrieve Top-K Chunks
    retrieved_chunks = []

    for idx in indices[0]:
        retrieved_chunks.append(
            all_recursive_chunks[idx]
        )

    context = "\n".join(retrieved_chunks)

    # Step 4: Create Prompt
    prompt = f"""
Context:
{context}

Question:
{question}

Answer:
"""

    # Step 5: Generate Response
    response = generator(
        prompt,
        max_new_tokens=150,
        do_sample=False
    )

    answer = response[0]["generated_text"]

    return answer, context

In [56]:
question = "How many casual leaves are available?"

answer, context = employee_policy_assistant(
    question
)

print("="*60)
print("QUESTION")
print("="*60)
print(question)

print("\n")

print("="*60)
print("RETRIEVED CONTEXT")
print("="*60)
print(context)

print("\n")

print("="*60)
print("FINAL ANSWER")
print("="*60)
print(answer)

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION
How many casual leaves are available?


RETRIEVED CONTEXT
This is a dummy content for Employee_Handbook.pdf
This is a dummy content for Leave_Policy.pdf
This is a dummy content for Travel_Policy.pdf


FINAL ANSWER

Context:
This is a dummy content for Employee_Handbook.pdf
This is a dummy content for Leave_Policy.pdf
This is a dummy content for Travel_Policy.pdf

Question:
How many casual leaves are available?

Answer:
There are 3 casual leaves available.


In [57]:
questions = [

    "How many casual leaves are available?",

    "Can employees work from home?",

    "How does travel reimbursement work?",

    "Who is covered under medical insurance?"
]

for question in questions:

    answer, context = employee_policy_assistant(
        question
    )

    print("\n")
    print("="*80)

    print("Employee Question:")
    print(question)

    print("\nRetrieved Context:")
    print(context)

    print("\nGenerated Answer:")
    print(answer)

    print("="*80)

[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)




Employee Question:
How many casual leaves are available?

Retrieved Context:
This is a dummy content for Employee_Handbook.pdf
This is a dummy content for Leave_Policy.pdf
This is a dummy content for Travel_Policy.pdf

Generated Answer:

Context:
This is a dummy content for Employee_Handbook.pdf
This is a dummy content for Leave_Policy.pdf
This is a dummy content for Travel_Policy.pdf

Question:
How many casual leaves are available?

Answer:
There are 3 casual leaves available.


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)




Employee Question:
Can employees work from home?

Retrieved Context:
This is a dummy content for Work_From_Home_Policy.pdf
This is a dummy content for Employee_Handbook.pdf
This is a dummy content for Leave_Policy.pdf

Generated Answer:

Context:
This is a dummy content for Work_From_Home_Policy.pdf
This is a dummy content for Employee_Handbook.pdf
This is a dummy content for Leave_Policy.pdf

Question:
Can employees work from home?

Answer:
Yes, employees can work from home.


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)




Employee Question:
How does travel reimbursement work?

Retrieved Context:
This is a dummy content for Travel_Policy.pdf
This is a dummy content for Medical_Insurance_Policy.pdf
This is a dummy content for Work_From_Home_Policy.pdf

Generated Answer:

Context:
This is a dummy content for Travel_Policy.pdf
This is a dummy content for Medical_Insurance_Policy.pdf
This is a dummy content for Work_From_Home_Policy.pdf

Question:
How does travel reimbursement work?

Answer:
Travel reimbursement is a policy that allows employees to reimburse themselves for travel expenses incurred for business purposes. This policy is designed to reduce the financial burden of travel expenses for employees and promote efficient use of company resources.

To reimburse travel expenses, employees must submit a travel expense report to their manager or HR representative. The report should include details such as the date of the trip, the purpose of the trip, the amount of the trip, and any necessary expenses s

In [58]:
test_questions = [

    "How many sick leaves are available?",

    "Can employees permanently work from home?",

    "When are travel expenses reimbursed?",

    "Are family members covered by medical insurance?"
]

In [59]:
expected_answers = [

    "Employees receive 15 sick leaves annually.",

    "Employees may work from home twice per week. Manager approval is required for additional remote work.",

    "Travel expenses are reimbursed within 30 days.",

    "Dependent coverage is available for spouse and children."
]

In [60]:
import time

results = []

for i, question in enumerate(test_questions):

    start_time = time.time()

    answer, context = employee_policy_assistant(
        question
    )

    end_time = time.time()

    response_time = end_time - start_time

    results.append([
        question,
        expected_answers[i],
        answer,
        round(response_time,2)
    ])

[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

In [61]:
import pandas as pd

evaluation_df = pd.DataFrame(
    results,
    columns=[
        "Question",
        "Expected Answer",
        "Generated Answer",
        "Response Time (sec)"
    ]
)

evaluation_df

,Question,Expected Answer,Generated Answer,Response Time (sec)
0,How many sick leaves are available?,Employees receive 15 sick leaves annually.,\nContext:\nThis is a dummy content for Medica...,24.03
1,Can employees permanently work from home?,Employees may work from home twice per week. M...,\nContext:\nThis is a dummy content for Work_F...,27.30
2,When are travel expenses reimbursed?,Travel expenses are reimbursed within 30 days.,\nContext:\nThis is a dummy content for Travel...,17.52
3,Are family members covered by medical insurance?,Dependent coverage is available for spouse and...,\nContext:\nThis is a dummy content for Medica...,15.88


In [62]:
correct = 0

total = len(test_questions)

for i, question in enumerate(test_questions):

    query_embedding = model.encode([question])

    distances, indices = index.search(
        np.array(query_embedding,
                 dtype="float32"),
        k=1
    )

    retrieved_chunk = all_recursive_chunks[
        indices[0][0]
    ]

    if any(
        word.lower() in retrieved_chunk.lower()
        for word in expected_answers[i].split()
    ):
        correct += 1

retrieval_accuracy = (
    correct / total
) * 100

print(
    "Retrieval Accuracy:",
    round(retrieval_accuracy,2),
    "%"
)

Retrieval Accuracy: 75.0 %


In [63]:
from sklearn.metrics.pairwise import cosine_similarity

relevance_scores = []

for i in range(len(test_questions)):

    expected_embedding = model.encode(
        [expected_answers[i]]
    )

    generated_embedding = model.encode(
        [results[i][2]]
    )

    score = cosine_similarity(
        expected_embedding,
        generated_embedding
    )[0][0]

    relevance_scores.append(score)

average_relevance = (
    sum(relevance_scores)
    / len(relevance_scores)
)

print(
    "Average Answer Relevance:",
    round(average_relevance,4)
)

Average Answer Relevance: 0.5016


In [64]:
avg_response_time = evaluation_df[
    "Response Time (sec)"
].mean()

print(
    "Average Response Time:",
    round(avg_response_time,2),
    "seconds"
)

Average Response Time: 21.18 seconds


In [65]:
print("="*60)
print("RAG SYSTEM EVALUATION REPORT")
print("="*60)

print(
    "\nRetrieval Accuracy :",
    round(retrieval_accuracy,2),
    "%"
)

print(
    "\nAverage Answer Relevance :",
    round(average_relevance,4)
)

print(
    "\nAverage Response Time :",
    round(avg_response_time,2),
    "seconds"
)

RAG SYSTEM EVALUATION REPORT

Retrieval Accuracy : 75.0 %

Average Answer Relevance : 0.5016

Average Response Time : 21.18 seconds
